# K-Nearest Neighbours — Implementations

The same classifier twice more. KNN has no parameters and no loss, so there is no gradient anywhere and autograd plays no role — the torch lane is about vectorising the distance computation, the sklearn lane about pinning the library to the scratch algorithm (`brute`, `euclidean`). Every lane ends on the same literal fixture: odd k, two classes, continuous data — no vote ties, no distance ties, so the three lanes must agree prediction for prediction, exactly.

## 07_knn_classifier

Memorise the training set; classify by majority vote of the k nearest points.

### torch

Nothing to differentiate — KNN is pure geometry, so this lane uses torch for tensor ops, not autograd. **What torch adds:** `torch.cdist` computes the entire (n_query, n_train) distance matrix in one call and `topk(largest=False)` selects the k nearest per row, replacing the scratch lane's per-query Python loop. The majority vote stays in NumPy, identical to the scratch lane, so tie-breaking cannot drift.

In [ ]:
import numpy as np
import torch

# hints:
# 1. torch.cdist(Xq, Xtr) is the whole (n_query, n_train) distance matrix in one call.
# 2. topk(k, largest=False) picks the k nearest per row, values already sorted.
# 3. Vote in NumPy exactly like the scratch lane, so tie-breaking cannot drift.
# 4. as_tensor on a float64 array keeps float64 — no .float() anywhere in this lane.


class KNNClassifierScratch:
    """K-Nearest Neighbors on tensors: one cdist call replaces the per-query
    Python loop, topk replaces argpartition. The vote stays in NumPy."""

    def __init__(self, n_neighbors=5):
        if n_neighbors < 1:
            raise ValueError("n_neighbors must be >= 1")
        self.n_neighbors = n_neighbors
        self.X_train_ = None
        self.y_train_ = None

    def fit(self, X, y):
        """Store the training set as a float64 tensor (still no computation)."""
        self.X_train_ = torch.as_tensor(np.asarray(X, dtype=float))
        self.y_train_ = np.asarray(y)
        if self.X_train_.ndim != 2:
            raise ValueError("X must be 2D")
        if self.X_train_.shape[0] != self.y_train_.shape[0]:
            raise ValueError("X and y must have the same number of samples")
        if self.n_neighbors > self.X_train_.shape[0]:
            raise ValueError("n_neighbors exceeds training set size")
        return self

    def predict(self, X):
        """All query distances at once, then a k-column topk and a NumPy vote."""
        Xq = torch.as_tensor(np.asarray(X, dtype=float))
        if Xq.ndim != 2 or Xq.shape[1] != self.X_train_.shape[1]:
            raise ValueError("X has incompatible feature shape")
        distances = torch.cdist(Xq, self.X_train_)
        _, nearest_idx = torch.topk(distances, self.n_neighbors, dim=1, largest=False)
        predictions = []
        for row in nearest_idx.numpy():
            labels, counts = np.unique(self.y_train_[row], return_counts=True)
            predictions.append(labels[np.argmax(counts)])
        return np.array(predictions)

    def score(self, X, y):
        """Classification accuracy."""
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: y_pred, acc_k5, knn_dist0
_rng_eq = np.random.default_rng(707)
X_tr_eq = np.vstack([_rng_eq.normal(-1.0, 1.0, size=(60, 4)),
                     _rng_eq.normal(1.0, 1.0, size=(60, 4))])
y_tr_eq = np.repeat(np.array([0, 1]), 60)
X_te_eq = _rng_eq.normal(0.0, 1.5, size=(40, 4))

_knn_eq = KNNClassifierScratch(n_neighbors=5).fit(X_tr_eq, y_tr_eq)
y_pred = [int(v) for v in _knn_eq.predict(X_te_eq)]
acc_k5 = _knn_eq.score(X_tr_eq, y_tr_eq)
_d0_eq = torch.cdist(torch.as_tensor(X_te_eq[:1]), torch.as_tensor(X_tr_eq))[0]
knn_dist0 = torch.topk(_d0_eq, 5, largest=False).values.tolist()
print("train acc (k=5):", acc_k5)
print("5 nearest distances to query 0:", np.round(knn_dist0, 4))


In [ ]:
# 1-NN memorises: every training point is its own nearest neighbour.
assert KNNClassifierScratch(n_neighbors=1).fit(X_tr_eq, y_tr_eq).score(X_tr_eq, y_tr_eq) == 1.0, \
    "1-NN must score 100% on its own training set"

# cdist is plain Euclidean distance, nothing fancier.
_Xq_t = torch.as_tensor(X_te_eq)
_Xt_t = torch.as_tensor(X_tr_eq)
_d_manual = torch.sqrt(((_Xq_t[:, None, :] - _Xt_t[None, :, :]) ** 2).sum(-1))
assert float(torch.max(torch.abs(torch.cdist(_Xq_t, _Xt_t) - _d_manual))) < 1e-10, \
    "cdist must equal the hand-rolled distance matrix"

# The vectorised cdist+topk pipeline agrees with a per-query argsort loop.
for _i in range(5):
    _d = np.linalg.norm(X_tr_eq - X_te_eq[_i], axis=1)
    _lab, _cnt = np.unique(y_tr_eq[np.argsort(_d)[:5]], return_counts=True)
    assert y_pred[_i] == int(_lab[np.argmax(_cnt)]), "vectorised vote = per-query vote"


### library

The estimator itself, with its search strategy pinned: `algorithm='brute', metric='euclidean'` makes sklearn do literally what the scratch loop does. **What the library adds:** the spatial indexes (`kd_tree`, `ball_tree`) it can swap in when brute force gets slow, plus `kneighbors()` for inspecting distances — here brute is forced so the comparison stays honest.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# hints:
# 1. algorithm='brute', metric='euclidean' pins sklearn to the scratch computation.
# 2. sklearn breaks vote ties toward the smallest label — same as np.unique + argmax.
# 3. Odd k on two classes means no vote ties, so agreement is exact rather than lucky.
# 4. kneighbors() hands back sorted distances and indices — perfect for the checks.


class KNNClassifierScratch:
    """sklearn's KNeighborsClassifier pinned to brute-force Euclidean search —
    the estimator the scratch class reimplements, with the same tie-breaking."""

    def __init__(self, n_neighbors=5):
        if n_neighbors < 1:
            raise ValueError("n_neighbors must be >= 1")
        self.n_neighbors = n_neighbors
        self._model = KNeighborsClassifier(n_neighbors=n_neighbors,
                                           algorithm="brute", metric="euclidean")

    def fit(self, X, y):
        """Delegate storage to sklearn; keep the arrays for the checks."""
        self.X_train_ = np.asarray(X, dtype=float)
        self.y_train_ = np.asarray(y)
        self._model.fit(self.X_train_, self.y_train_)
        return self

    def predict(self, X):
        """Majority vote of the k nearest, done by the library."""
        return self._model.predict(np.asarray(X, dtype=float))

    def score(self, X, y):
        """Classification accuracy."""
        return float(self._model.score(np.asarray(X, dtype=float), np.asarray(y)))


In [ ]:
# exports: y_pred, acc_k5, knn_dist0
_rng_eq = np.random.default_rng(707)
X_tr_eq = np.vstack([_rng_eq.normal(-1.0, 1.0, size=(60, 4)),
                     _rng_eq.normal(1.0, 1.0, size=(60, 4))])
y_tr_eq = np.repeat(np.array([0, 1]), 60)
X_te_eq = _rng_eq.normal(0.0, 1.5, size=(40, 4))

_knn_eq = KNNClassifierScratch(n_neighbors=5).fit(X_tr_eq, y_tr_eq)
y_pred = [int(v) for v in _knn_eq.predict(X_te_eq)]
acc_k5 = _knn_eq.score(X_tr_eq, y_tr_eq)
_d0_eq, _ = _knn_eq._model.kneighbors(X_te_eq[:1], n_neighbors=5)
knn_dist0 = _d0_eq[0].tolist()
print("train acc (k=5):", acc_k5)
print("5 nearest distances to query 0:", np.round(knn_dist0, 4))


In [ ]:
# 1-NN memorises the training set through the library, too.
assert KNNClassifierScratch(n_neighbors=1).fit(X_tr_eq, y_tr_eq).score(X_tr_eq, y_tr_eq) == 1.0, \
    "1-NN must score 100% on its own training set"

# Brute-force kneighbors returns exactly the sorted Euclidean distances.
_d_sk, _ = _knn_eq._model.kneighbors(X_te_eq, n_neighbors=5)
_d_np = np.sort(np.linalg.norm(X_te_eq[:, None, :] - X_tr_eq[None, :, :], axis=2), axis=1)
assert np.max(np.abs(_d_sk - _d_np[:, :5])) < 1e-10, "kneighbors = plain Euclidean, sorted"

# Euclidean KNN is translation-invariant: shift everything, nothing changes.
_y_shift = KNNClassifierScratch(n_neighbors=5).fit(X_tr_eq + 3.0, y_tr_eq).predict(X_te_eq + 3.0)
assert [int(v) for v in _y_shift] == y_pred, "shifting all points must not change any vote"
